## Child Malnutrition
Analyzing child malnutrition and mortality trends across Sub-Saharan Africa using WHO data, with fairness auditing and causal analysis planned.

In [1]:
import pandas as pd 
import requests 
from io import StringIO
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
# Connecting to WHO API to get data
url = "https://ghoapi.azureedge.net/api" 
response = requests.get(f"{url}/Indicator")
print("Connection status: ", response.status_code ) 

Connection status:  200


In [3]:
# Checking to see all indicators available

indicator_list = response.json()['value']
df = pd.DataFrame(indicator_list)
 
print(len(df))

df[['IndicatorCode', 'IndicatorName']].head(10)

3068


,IndicatorCode,IndicatorName
0,GOE_Q203,Lack of integration between various health ser...
1,HCF_REL_ELECTRICITY,Percentage of health-care facilities with reli...
2,HIV_0000000007,"Testing and counselling facilities, reported n..."
3,HIV_0000000008,"Testing and counselling facilities, estimated ..."
4,EMFLIMITPOWERDENSITY,Power density (W/m^2)
5,EMFSAR,Specific absorption rate (SAR) (W/kg)
6,EMFSTATUS,Legislative status
7,EQ_OVERWEIGHTADULT,Prevalence of overweight and obesity
8,FINPROTECTION_CATA_TOT_10_LEVEL_MILLION,Population with household spending on health g...
9,FINPROTECTION_CATA_TOT_10_LEVEL_SH,Population with household spending on health g...


In [4]:
# Search for nutrition indicators we might be interested in. 

keywords = 'child|children|stunting|wasting|underweight|overweight|breastfeed|anaemia|malnutrition|mortality'
 
interests = df[
    df['IndicatorName'].str.lower().str.contains(keywords, na=False)
][['IndicatorCode', 'IndicatorName']]
 
print("Related indicators found: ", len(interests))

interests  
 

Related indicators found:  216


,IndicatorCode,IndicatorName
7,EQ_OVERWEIGHTADULT,Prevalence of overweight and obesity
11,HIV_0000000015,Number of pregnant women living with HIV who r...
12,HIV_0000000016,Number of pregnant women living with HIV who r...
85,HIV_0000000011,Reported number of children receiving antiretr...
86,HIV_0000000012,Reported number of children receiving antiretr...
...,...,...
3026,CHILDDEATH5TO14,Number of deaths among children 5 to 14 years ...
3030,NCD_CCS_OvwtPlan,Existence of operational policy/strategy/actio...
3039,NTD_LEPR13,New leprosy child case detection rate (less th...
3058,NUTRITION_ANAEMIA_PREGNANT_PREV,Prevalence of anaemia in pregnant women (aged ...


In [5]:
# These are the 10 malnutrition indicators we are interested in, with their corresponding column names for the final dataset.
indicators = { 
    "NUTRITION_ANT_HAZ_NE2":  "stunting_pct",                    # stunting
    "NUTRITION_ANT_WHZ_NE2":  "wasting_pct",                     # wasting
    "NUTRITION_ANT_WAZ_NE2":  "underweight_pct",                 # underweight
    "NUTRITION_ANT_WHZ_PE2":  "overweight_pct",                  # overweight
    "NUTRITION_ANT_WHZ_NE3":  "severe_wasting_pct",              # severe wasting
    "NUTRITION_2005":          "min_acceptable_diet_pct",        # diet quality
    "WHOSIS_000001":           "under_five_mortality_rate",      # Under-5 mortality rate
    "WHOSIS_000002":           "infant_mortality_rate",          # Infant mortality rate
    "WHOSIS_000003":           "neonatal_mortality_rate",        # Neonatal mortality rate
    "NUTRITION_570":           "early_initiation_breastfeeding", # Early initiation of breastfeeding 
}

# Below are the Sub-Saharan Africa countries and their country_code
ssa_countries = {
    "AGO","BEN","BWA","BFA","BDI","CPV","CMR","CAF","TCD","COM",
    "COD","COG","CIV","DJI","GNQ","ERI","SWZ","ETH","GAB","GMB",
    "GHA","GIN","GNB","KEN","LSO","LBR","MDG","MWI","MLI","MRT",
    "MUS","MOZ","NAM","NER","NGA","RWA","STP","SEN","SLE","SOM",
    "ZAF","SSD","SDN","TZA","TGO","UGA","ZMB","ZWE"
}
 
print("Indicators to download :", len(indicators))
print("SSA countries included :",len(ssa_countries))
 

Indicators to download : 10
SSA countries included : 48


In [6]:
# Function to fetch data for a given indicator code and return a DataFrame with the relevant columns.

def get_indicator(code, column_name):
 
    url1 = f"{url}/{code}"
    print(column_name, end=" ", flush=True)
 
    response = requests.get(url1, timeout=30)

    # Checking the response is valid before parsing
    if response.status_code != 200 or not response.text.strip():
        print(response.status_code)
        return pd.DataFrame()
    indi_records = response.json().get("value", [])
 
    # Keeping only SSA country rows that has a numeric value
    rows = []
    for r in indi_records:
        country_code = r.get("SpatialDim", "")
        value        = r.get("NumericValue")
        if country_code in ssa_countries and value is not None and 2010 <= r.get("TimeDim", 0) <= 2026:
            rows.append({
                "country_code":      country_code,
                "year":      r.get("TimeDim"),
                column_name: value,
            })
 
    df = pd.DataFrame(rows)
    print(len(df), "rows")
    return df
 


In [7]:
# Function to download all indicators and merge 

def data_load():
 
    data = []
    for code, name in indicators.items():
        df = get_indicator(code, name)
        if not df.empty:
            data.append(df)
 
    if not data:
        print("\nNo data retrieved.")
        return pd.DataFrame()
 
    # Merge all indicators on country_code + year
    merged = data[0]
    for df in data[1:]:
        col = [c for c in df.columns if c not in ("country_code", "year")]
        merged = merged.merge(df[["country_code", "year"] + col],
                              on=["country_code", "year"], how="outer")
 
    merged = merged.sort_values(["country_code", "year"]).reset_index(drop=True)
 
    print("\n  Rows     :", len(merged))
    print("  Countries: ", merged['country_code'].nunique())
    print("  Years    : ", int(merged['year'].min()), ",", int(merged['year'].max()))
    print("  Columns  : ", list(merged.columns))
    return merged
 
 

In [8]:
df = data_load()
df.head(10)

stunting_pct 16899 rows
wasting_pct 15915 rows
underweight_pct 404
overweight_pct 404
severe_wasting_pct 404
min_acceptable_diet_pct 51 rows
under_five_mortality_rate 1728 rows
infant_mortality_rate 1728 rows
neonatal_mortality_rate 672 rows
early_initiation_breastfeeding 77 rows

  Rows     : 11738663
  Countries:  48
  Years    :  2010 , 2024
  Columns  :  ['country_code', 'year', 'stunting_pct', 'wasting_pct', 'min_acceptable_diet_pct', 'under_five_mortality_rate', 'infant_mortality_rate', 'neonatal_mortality_rate', 'early_initiation_breastfeeding']


,country_code,year,stunting_pct,wasting_pct,min_acceptable_diet_pct,under_five_mortality_rate,infant_mortality_rate,neonatal_mortality_rate,early_initiation_breastfeeding
0,AGO,2010,NaN,NaN,NaN,60.071615,51.486706,36.197696,NaN
1,AGO,2010,NaN,NaN,NaN,60.071615,50.368592,36.197696,NaN
2,AGO,2010,NaN,NaN,NaN,60.071615,49.261258,36.197696,NaN
3,AGO,2010,NaN,NaN,NaN,55.861059,51.486706,36.197696,NaN
4,AGO,2010,NaN,NaN,NaN,55.861059,50.368592,36.197696,NaN
5,AGO,2010,NaN,NaN,NaN,55.861059,49.261258,36.197696,NaN
6,AGO,2010,NaN,NaN,NaN,57.964611,51.486706,36.197696,NaN
7,AGO,2010,NaN,NaN,NaN,57.964611,50.368592,36.197696,NaN
8,AGO,2010,NaN,NaN,NaN,57.964611,49.261258,36.197696,NaN
9,AGO,2011,NaN,NaN,NaN,56.606110,49.925589,34.983319,NaN


In [9]:
# Checking for duplicates per country-year since the rows loads are over 11,000,000 

duplicate = df.groupby(['country_code', 'year']).size().reset_index(name='count')
print("Unique country-year combos: ", len(duplicate)) 
print("Rows with duplicates (>1 per country-year):", (duplicate['count'] > 1).sum())
print("\nMax duplicates for a single country-year:", duplicate['count'].max())
print("Average rows per country-year:", f"{duplicate['count'].mean():.2f}")


Unique country-year combos:  676
Rows with duplicates (>1 per country-year): 597

Max duplicates for a single country-year: 99225
Average rows per country-year: 17364.89


In [10]:
# Showing an example of a country-year with duplicates, if any exist

if (duplicate['count'] > 1).any():
    example = duplicate[duplicate['count'] > 1].iloc[0]
    country, year = example['country_code'], example['year']
    print(f"\nExample: {country}, {year} - {example['count']} rows")
    display(df[(df['country_code'] == country) & (df['year'] == year)].head())



Example: AGO, 2010 - 9 rows


,country_code,year,stunting_pct,wasting_pct,min_acceptable_diet_pct,under_five_mortality_rate,infant_mortality_rate,neonatal_mortality_rate,early_initiation_breastfeeding
0,AGO,2010,NaN,NaN,NaN,60.071615,51.486706,36.197696,NaN
1,AGO,2010,NaN,NaN,NaN,60.071615,50.368592,36.197696,NaN
2,AGO,2010,NaN,NaN,NaN,60.071615,49.261258,36.197696,NaN
3,AGO,2010,NaN,NaN,NaN,55.861059,51.486706,36.197696,NaN
4,AGO,2010,NaN,NaN,NaN,55.861059,50.368592,36.197696,NaN


Same country-year showing different mortality rates, like 57.96 versus 60.07 for under-five mortality, and infant mortality bouncing between 51.49, 50.37, and 49.26.

These are probably different estimates or subgroups, things like male versus female, urban versus rural, or different age brackets within the indicator.

Since the goal here is trend analysis and correlations, we are going to take the mean, just a simple average across whatever estimates are available for that country-year.

In [11]:
# Aggregate by taking the mean for each country-year

df_clean = df.groupby(['country_code', 'year'], as_index=False).mean()

print(f"Original rows: {len(df)}")
print(f"After aggregation: {len(df_clean)}")
print(f"Countries: {df_clean['country_code'].nunique()}")
print(f"Year range: {df_clean['year'].min()} – {df_clean['year'].max()}")


Original rows: 11738663
After aggregation: 676
Countries: 48
Year range: 2010 – 2024


In [12]:
# Checking the result of AGO to confirm as that was the earlier example we looked at
print("\nExample AGO 2010 after aggregation:")
display(df_clean[(df_clean['country_code'] == 'AGO') & (df_clean['year'] == 2010)])


Example AGO 2010 after aggregation:


,country_code,year,stunting_pct,wasting_pct,min_acceptable_diet_pct,under_five_mortality_rate,infant_mortality_rate,neonatal_mortality_rate,early_initiation_breastfeeding
0,AGO,2010,NaN,NaN,NaN,57.965762,50.372185,36.197696,NaN


In [13]:
# Checking the data coverage for each column to know the missing values and the percentage of coverage for each column in the cleaned dataset.
coverage = df_clean.isnull().sum()
total_rows = len(df_clean)
coverage_pct = (1 - coverage / total_rows) * 100

coverage_df = pd.DataFrame({
    'Missing': coverage,
    'Non-Missing': total_rows - coverage,
    'Coverage %': coverage_pct
}).sort_values('Coverage %', ascending=False)

print("Column coverage:\n")
display(coverage_df)

Column coverage:



,Missing,Non-Missing,Coverage %
country_code,0,676,100.000000
year,0,676,100.000000
neonatal_mortality_rate,4,672,99.408284
under_five_mortality_rate,100,576,85.207101
infant_mortality_rate,100,576,85.207101
stunting_pct,466,210,31.065089
wasting_pct,474,202,29.881657
early_initiation_breastfeeding,599,77,11.390533
min_acceptable_diet_pct,625,51,7.544379


In [14]:
# Dropping columns with low coverage 

columns_drop = ['min_acceptable_diet_pct', 'early_initiation_breastfeeding']
df_clean = df_clean.drop(columns=columns_drop)

In [15]:
print("Columns after dropping low-coverage ones:", list(df_clean.columns))
print("Shape:", df_clean.shape)

Columns after dropping low-coverage ones: ['country_code', 'year', 'stunting_pct', 'wasting_pct', 'under_five_mortality_rate', 'infant_mortality_rate', 'neonatal_mortality_rate']
Shape: (676, 7)


In [16]:
# the Interpolating of missing values will be done by country

df_interpolate = df_clean.copy()

# each column is going to be interpolated and  grouped by country
cols_interpolate = ['stunting_pct', 'wasting_pct', 'under_five_mortality_rate', 
                       'infant_mortality_rate', 'neonatal_mortality_rate']

for col in cols_interpolate:
    df_interpolate[col] = df_interpolate.groupby('country_code')[col].transform(
        lambda x: x.interpolate(method='linear', limit_direction='both')
    )

#printing the missing values after interpolation to know if there are any remaining missing values after the interpolation process.
print(df_interpolate[cols_interpolate].isnull().sum())

stunting_pct                 42
wasting_pct                  42
under_five_mortality_rate     0
infant_mortality_rate         0
neonatal_mortality_rate       0
dtype: int64


In [17]:
print("\nValue ranges after interpolation:")
for col in cols_interpolate:
    print(f"{col}: {df_interpolate[col].min():.2f} to {df_interpolate[col].max():.2f}")


Value ranges after interpolation:
stunting_pct: 6.29 to 56.91
wasting_pct: 0.09 to 15.98
under_five_mortality_rate: 43.92 to 74.90
infant_mortality_rate: 39.24 to 65.31
neonatal_mortality_rate: 6.76 to 46.69


In [18]:
#finding the countries that still have missing values for stunting_pct after interpolation and printing the number of affected countries and their country codes.
missing_stunting = df_interpolate[df_interpolate['stunting_pct'].isnull()]
countries_with_issue = missing_stunting['country_code'].unique()

print("Countries affected:", {len(countries_with_issue)})
print("Which countries:", sorted(countries_with_issue))

Countries affected: {3}
Which countries: ['BWA', 'MUS', 'SOM']


In [19]:
# Dropping countries with missing data after interpolation.

drop_coun = ['BWA', 'MUS', 'SOM']
final_df = df_interpolate[~df_interpolate['country_code'].isin(drop_coun)].copy()

print("Countries remaining:", {final_df['country_code'].nunique()})

Countries remaining: {45}
